# 🧬🔬 Biohub Cell Tracking V2 — Metric-Aware 3D+Time Lineage Baseline

This notebook is a stronger, conservative rewrite of the first working submission. It keeps the solution **offline, lightweight and Kaggle-submission safe**, but changes the scoring strategy:

| Component | V1 issue | V2 correction |
|---|---|---|
| Detection | many weak/border peaks and strided sampling artifacts | **full-Z + XY block-mean**, Otsu/relative threshold, local peak detection, centroid refinement |
| Node quality | isolated false positives increase the node-count penalty | **track-incident pruning** removes detections that never participate in an edge |
| Linking | loose edges can inflate FP | physical Hungarian assignment with a tighter gate and deterministic pruning |
| Division | overly permissive split edges hurt precision | **conservative mitosis pass**: parent already has one child, second daughter is close to both parent and sister |
| Submission | valid, but hard to tune | one CONFIG block, profile knobs, dataset-level diagnostics and strict validation |

The metric rewards correct graph edges and division structure, while penalizing excessive node overprediction. The key design choice is therefore **not simply detecting more bright spots**, but detecting stable cell centers that can form plausible tracks.

In [1]:

from __future__ import annotations

import gc
import glob
import json
import math
import os
import time
from collections import Counter, defaultdict, deque
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import blosc2
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

try:
    from skimage.feature import peak_local_max
    from skimage.filters import threshold_otsu
    SKIMAGE_AVAILABLE = True
except Exception:
    peak_local_max = None
    threshold_otsu = None
    SKIMAGE_AVAILABLE = False

np.random.seed(42)

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
CANDIDATE_TEST_DIRS = [
    Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/test'),
    Path('/kaggle/input/biohub-cell-tracking-during-development/test'),
]
TEST_DIR = next((p for p in CANDIDATE_TEST_DIRS if p.exists()), None)
if TEST_DIR is None:
    hits = [Path(p) for p in glob.glob('/kaggle/input/**/test', recursive=True)]
    TEST_DIR = next((p for p in hits if list(p.glob('*.zarr'))), CANDIDATE_TEST_DIRS[0])

OUTPUT_PATH = Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else Path('submission.csv')

# ---------------------------------------------------------------------
# Metric scale: Z, Y, X micrometers per voxel.
# ---------------------------------------------------------------------
SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)

# ---------------------------------------------------------------------
# Main profile. The defaults are intentionally precision-biased versus V1.
# Override in Kaggle by setting environment variables before running.
# ---------------------------------------------------------------------
PROFILE = os.environ.get('BIOHUB_PROFILE', 'v2_precision')

XY_DS = int(os.environ.get('BIOHUB_XY_DS', '4'))              # XY block mean: 4*0.40625 ≈ 1.625 µm, i.e. physically isotropic with Z.
SMOOTH_SIGMA = float(os.environ.get('BIOHUB_SMOOTH_SIGMA', '0.95'))
MIN_PEAK_DIST = int(os.environ.get('BIOHUB_MIN_PEAK_DIST', '3'))
THRESH_REL = float(os.environ.get('BIOHUB_THRESH_REL', '0.34'))
MIN_REL_CONTRAST = float(os.environ.get('BIOHUB_MIN_REL_CONTRAST', '0.08'))

# Border filter: keep strong border detections, drop weak ones. This specifically targets V1's z=0/y-edge artifacts.
BORDER_Z = int(os.environ.get('BIOHUB_BORDER_Z', '1'))
BORDER_YX = int(os.environ.get('BIOHUB_BORDER_YX', '2'))
BORDER_KEEP_QUANTILE = float(os.environ.get('BIOHUB_BORDER_KEEP_QUANTILE', '0.70'))

# Physical duplicate suppression after refinement.
NMS_RADIUS_UM = float(os.environ.get('BIOHUB_NMS_RADIUS_UM', '2.65'))

# Count stabilizer. It only cuts extreme frame-to-frame explosions.
MAX_FRAME_COUNT_MULT = float(os.environ.get('BIOHUB_MAX_FRAME_COUNT_MULT', '1.70'))
MAX_FRAME_COUNT_ADD = int(os.environ.get('BIOHUB_MAX_FRAME_COUNT_ADD', '45'))
MAX_NODES_PER_FRAME = int(os.environ.get('BIOHUB_MAX_NODES_PER_FRAME', '20000'))

# Linking and divisions.
MAX_LINK_DIST_UM = float(os.environ.get('BIOHUB_MAX_LINK_DIST_UM', '11.0'))
DETECT_DIVISIONS = os.environ.get('BIOHUB_DETECT_DIVISIONS', '1') != '0'
DIV_PARENT_DIST_UM = float(os.environ.get('BIOHUB_DIV_PARENT_DIST_UM', '8.75'))
DIV_SISTER_DIST_UM = float(os.environ.get('BIOHUB_DIV_SISTER_DIST_UM', '6.25'))
DIV_MIN_COUNT_GAIN = int(os.environ.get('BIOHUB_DIV_MIN_COUNT_GAIN', '1'))

# Remove unlinked single-frame detections. This is usually positive for the adjusted edge Jaccard node-count penalty.
PRUNE_ISOLATED_NODES = os.environ.get('BIOHUB_PRUNE_ISOLATED_NODES', '1') != '0'
KEEP_STRONG_ISOLATED = os.environ.get('BIOHUB_KEEP_STRONG_ISOLATED', '0') == '1'
STRONG_ISOLATED_QUANTILE = float(os.environ.get('BIOHUB_STRONG_ISOLATED_QUANTILE', '0.97'))

print('Biohub Cell Tracking V2 Metric-Aware Baseline')
print('TEST_DIR:', TEST_DIR, 'exists:', TEST_DIR.exists())
print('PROFILE:', PROFILE, '| skimage:', SKIMAGE_AVAILABLE)
print(f'Detection: XY_DS={XY_DS}, sigma={SMOOTH_SIGMA}, min_peak_dist={MIN_PEAK_DIST}, thresh_rel={THRESH_REL}')
print(f'Linking: max_link={MAX_LINK_DIST_UM}µm, divisions={DETECT_DIVISIONS}, prune_isolated={PRUNE_ISOLATED_NODES}')


Biohub Cell Tracking V2 Metric-Aware Baseline
TEST_DIR: /kaggle/input/competitions/biohub-cell-tracking-during-development/test exists: True
PROFILE: v2_precision | skimage: True
Detection: XY_DS=4, sigma=0.95, min_peak_dist=3, thresh_rel=0.34
Linking: max_link=11.0µm, divisions=True, prune_isolated=True


## 1 · Zarr streaming loader

The competition data is chunked one timepoint at a time. We stream a single `(Z, Y, X)` volume into memory, process it, and immediately release it. This keeps the notebook stable under the 12-hour Kaggle CPU/GPU notebook limit.

In [2]:

def dataset_names_from_test_dir(test_dir: Path = TEST_DIR) -> List[str]:
    if not test_dir.exists():
        raise FileNotFoundError(f'Test directory not found: {test_dir}')
    names = sorted(p.name[:-5] for p in test_dir.iterdir() if p.name.endswith('.zarr'))
    if not names:
        raise FileNotFoundError(f'No .zarr test datasets found in {test_dir}')
    return names


def read_array_meta(zarr_path: Path) -> Tuple[Path, Tuple[int, ...], np.dtype, Tuple[int, ...]]:
    """Return array_dir, shape, dtype, chunk_shape for the 4D image array."""
    # Official layout: dataset.zarr/0/zarr.json, chunks at 0/c/t/0/0/0
    candidates = [zarr_path / '0' / 'zarr.json', zarr_path / '0' / '.zarray']
    candidates += list(zarr_path.rglob('zarr.json'))[:8]
    candidates += list(zarr_path.rglob('.zarray'))[:8]

    seen = set()
    for meta_path in candidates:
        if meta_path in seen or not meta_path.exists():
            continue
        seen.add(meta_path)
        try:
            meta = json.load(open(meta_path, 'r'))
        except Exception:
            continue

        shape = tuple(meta.get('shape', ()))
        if len(shape) != 4:
            continue

        dtype_name = meta.get('data_type', meta.get('dtype'))
        dtype = np.dtype(dtype_name)

        if 'chunk_grid' in meta:
            chunk_shape = tuple(meta['chunk_grid']['configuration']['chunk_shape'])
        else:
            chunk_shape = tuple(meta.get('chunks', shape))

        return meta_path.parent, shape, dtype, chunk_shape

    raise FileNotFoundError(f'Could not find 4D zarr metadata under {zarr_path}')


def chunk_path_candidates(array_dir: Path, t: int) -> List[Path]:
    return [
        array_dir / 'c' / str(t) / '0' / '0' / '0',
        array_dir / f'{t}.0.0.0',
        array_dir / str(t) / '0' / '0' / '0',
    ]


def load_volume(array_dir: Path, shape: Tuple[int, ...], dtype: np.dtype, t: int) -> np.ndarray:
    """Load one timepoint as (Z, Y, X)."""
    for chunk_path in chunk_path_candidates(array_dir, t):
        if chunk_path.exists():
            raw = open(chunk_path, 'rb').read()
            try:
                dec = blosc2.decompress(raw)
            except Exception:
                dec = raw
            arr = np.frombuffer(dec, dtype=dtype)
            expected = int(np.prod(shape[1:]))
            if arr.size < expected:
                out = np.zeros(expected, dtype=dtype)
                out[:arr.size] = arr
                arr = out
            return arr[:expected].reshape(shape[1:])
    raise FileNotFoundError(f'Missing chunk for t={t} in {array_dir}')


def list_datasets() -> List[str]:
    names = dataset_names_from_test_dir(TEST_DIR)
    print(f'Found {len(names)} test datasets:', names)
    return names


## 2 · Detection: full-Z, XY block-mean, local maxima

The evaluation matches centroids using physical distance. Since `z=1.625µm` and `y=x=0.40625µm`, downsampling Z is expensive, while averaging XY by 4 creates an approximately isotropic grid. V2 therefore keeps full Z, denoises XY, finds local intensity peaks, refines each centroid on the original image, and removes weak boundary artifacts.

In [3]:

def block_mean_xy(vol: np.ndarray, factor: int = XY_DS) -> np.ndarray:
    """Average-pool XY by factor while preserving Z resolution."""
    Z, Y, X = vol.shape
    Y2, X2 = (Y // factor) * factor, (X // factor) * factor
    x = vol[:, :Y2, :X2].astype(np.float32, copy=False)
    return x.reshape(Z, Y2 // factor, factor, X2 // factor, factor).mean(axis=(2, 4))


def robust_threshold(sm: np.ndarray) -> Tuple[float, float, float]:
    """Otsu + relative-rise threshold. Returns threshold, background, dynamic range."""
    bg = float(np.median(sm))
    hi = float(np.percentile(sm, 99.9))
    dyn = max(hi - bg, 1e-6)
    rel_thr = bg + THRESH_REL * dyn
    try:
        otsu = float(threshold_otsu(sm)) if SKIMAGE_AVAILABLE else float(np.percentile(sm, 96.0))
    except Exception:
        otsu = float(np.percentile(sm, 96.0))
    # Otsu can be too low on sparse bright blobs; the relative-rise floor is the safety valve.
    return max(otsu, rel_thr), bg, dyn


def fallback_peak_local_max(sm: np.ndarray, threshold_abs: float, min_distance: int) -> np.ndarray:
    """Small dependency-free local maxima fallback."""
    size = 2 * int(min_distance) + 1
    mx = maximum_filter(sm, size=(size, size, size), mode='nearest')
    mask = (sm >= mx) & (sm > threshold_abs)
    coords = np.argwhere(mask)
    if coords.size == 0:
        return coords.astype(np.int32)
    vals = sm[coords[:, 0], coords[:, 1], coords[:, 2]]
    return coords[np.argsort(-vals)].astype(np.int32)


def physical_nms(coords_vox: np.ndarray, scores: np.ndarray, radius_um: float = NMS_RADIUS_UM) -> Tuple[np.ndarray, np.ndarray]:
    if len(coords_vox) <= 1:
        return coords_vox, scores
    pts = coords_vox.astype(np.float64) * SCALE[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    suppressed = np.zeros(len(coords_vox), dtype=bool)
    keep = []
    for i in order:
        if suppressed[i]:
            continue
        keep.append(i)
        for j in tree.query_ball_point(pts[i], r=radius_um):
            suppressed[j] = True
    keep = np.array(keep, dtype=np.int64)
    return coords_vox[keep], scores[keep]


def refine_centroid(vol: np.ndarray, approx_zyx: np.ndarray) -> Tuple[np.ndarray, float]:
    """Local intensity-weighted refinement in original voxel coordinates."""
    Z, Y, X = vol.shape
    z, y, x = [int(round(v)) for v in approx_zyx]
    rz, ry, rx = 2, 5, 5
    z0, z1 = max(0, z - rz), min(Z, z + rz + 1)
    y0, y1 = max(0, y - ry), min(Y, y + ry + 1)
    x0, x1 = max(0, x - rx), min(X, x + rx + 1)
    crop = vol[z0:z1, y0:y1, x0:x1].astype(np.float32, copy=False)
    if crop.size == 0:
        return np.array([z, y, x], dtype=np.float64), 0.0

    bg = float(np.percentile(crop, 20.0))
    weights = crop - bg
    weights[weights < 0] = 0
    total = float(weights.sum())
    if total <= 1e-6:
        loc = np.unravel_index(int(np.argmax(crop)), crop.shape)
        return np.array([z0 + loc[0], y0 + loc[1], x0 + loc[2]], dtype=np.float64), float(crop[loc])

    zz, yy, xx = np.indices(crop.shape)
    refined = np.array([
        z0 + float((zz * weights).sum() / total),
        y0 + float((yy * weights).sum() / total),
        x0 + float((xx * weights).sum() / total),
    ], dtype=np.float64)
    return refined, float(weights.max())


def detect_cells(vol: np.ndarray, prev_count: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray]:
    """Return integer centroid coordinates (z,y,x) and detector scores."""
    Z, Y, X = vol.shape
    ds = block_mean_xy(vol, XY_DS)
    sm = gaussian_filter(ds, sigma=SMOOTH_SIGMA, mode='nearest')
    threshold_abs, bg, dyn = robust_threshold(sm)

    if SKIMAGE_AVAILABLE:
        coords_ds = peak_local_max(
            sm,
            min_distance=MIN_PEAK_DIST,
            threshold_abs=threshold_abs,
            exclude_border=False,
        ).astype(np.int32)
    else:
        coords_ds = fallback_peak_local_max(sm, threshold_abs, MIN_PEAK_DIST)

    if coords_ds.size == 0:
        # Safe fallback: a few highest voxels in the smoothed image, never an empty dataset frame.
        flat = np.argpartition(sm.ravel(), -3)[-3:]
        coords_ds = np.array(np.unravel_index(flat, sm.shape)).T.astype(np.int32)

    # Score candidates on the smoothed isotropic grid.
    peak_scores = sm[coords_ds[:, 0], coords_ds[:, 1], coords_ds[:, 2]].astype(np.float32)
    rel_contrast = (peak_scores - bg) / max(dyn, 1e-6)
    keep = rel_contrast >= MIN_REL_CONTRAST
    coords_ds, peak_scores = coords_ds[keep], peak_scores[keep]

    if len(coords_ds) == 0:
        return np.empty((0, 3), dtype=np.int32), np.empty((0,), dtype=np.float32)

    # Map from XY-block grid back to original coords; Z is unchanged.
    approx = coords_ds.astype(np.float64)
    approx[:, 1] = approx[:, 1] * XY_DS + (XY_DS - 1) / 2.0
    approx[:, 2] = approx[:, 2] * XY_DS + (XY_DS - 1) / 2.0

    refined, refined_scores = [], []
    for a, s in zip(approx, peak_scores):
        r, rs = refine_centroid(vol, a)
        refined.append(r)
        refined_scores.append(max(float(s), rs))
    coords = np.vstack(refined).astype(np.float64)
    scores = np.array(refined_scores, dtype=np.float32)

    # Remove weak boundary peaks while preserving high-confidence border cells.
    if len(coords):
        cz, cy, cx = coords[:, 0], coords[:, 1], coords[:, 2]
        border = (
            (cz <= BORDER_Z) | (cz >= (Z - 1 - BORDER_Z)) |
            (cy <= BORDER_YX) | (cy >= (Y - 1 - BORDER_YX)) |
            (cx <= BORDER_YX) | (cx >= (X - 1 - BORDER_YX))
        )
        border_score_floor = float(np.quantile(scores, BORDER_KEEP_QUANTILE)) if len(scores) > 8 else -np.inf
        keep = (~border) | (scores >= border_score_floor)
        coords, scores = coords[keep], scores[keep]

    coords, scores = physical_nms(coords, scores, NMS_RADIUS_UM)

    # Count stabilizer: only trims implausible explosions relative to adjacent frames.
    if prev_count is not None and prev_count >= 8 and len(coords) > prev_count * MAX_FRAME_COUNT_MULT + MAX_FRAME_COUNT_ADD:
        cap = int(prev_count * MAX_FRAME_COUNT_MULT + MAX_FRAME_COUNT_ADD)
        order = np.argsort(-scores)[:cap]
        coords, scores = coords[order], scores[order]

    if len(coords) > MAX_NODES_PER_FRAME:
        order = np.argsort(-scores)[:MAX_NODES_PER_FRAME]
        coords, scores = coords[order], scores[order]

    coords = np.rint(coords).astype(np.int32)
    if len(coords):
        coords[:, 0] = np.clip(coords[:, 0], 0, Z - 1)
        coords[:, 1] = np.clip(coords[:, 1], 0, Y - 1)
        coords[:, 2] = np.clip(coords[:, 2], 0, X - 1)
    return coords, scores.astype(np.float32)


## 3 · Tracking and conservative division detection

Primary links are one-to-one Hungarian assignments in physical space. A second pass adds a division edge only when a parent already has one daughter and an unmatched second daughter is physically close to both the parent and the first daughter. This protects division precision while still giving the metric a chance to reward mitosis structure.

In [4]:

def link_frames(prev_ids: List[int], prev_xyz: np.ndarray, curr_ids: List[int], curr_xyz: np.ndarray) -> List[Tuple[int, int]]:
    """Frame-to-frame links. A source may receive a second outgoing edge only through the division pass."""
    if len(prev_ids) == 0 or len(curr_ids) == 0:
        return []

    P = prev_xyz.astype(np.float64) * SCALE[None, :]
    C = curr_xyz.astype(np.float64) * SCALE[None, :]
    D = np.sqrt(((P[:, None, :] - C[None, :, :]) ** 2).sum(axis=2))

    BIG = 1e6
    cost = np.where(D <= MAX_LINK_DIST_UM, D, BIG)
    ri, ci = linear_sum_assignment(cost)

    edges: List[Tuple[int, int]] = []
    parent_children: Dict[int, List[int]] = defaultdict(list)  # prev index -> curr indices
    matched_curr = set()

    for r, c in zip(ri, ci):
        if cost[r, c] < BIG:
            edges.append((int(prev_ids[r]), int(curr_ids[c])))
            parent_children[int(r)].append(int(c))
            matched_curr.add(int(c))

    # Conservative mitosis pass. Avoid creating divisions unless there is count pressure.
    if DETECT_DIVISIONS and (len(curr_ids) - len(prev_ids) >= DIV_MIN_COUNT_GAIN):
        for c in range(len(curr_ids)):
            if c in matched_curr:
                continue
            best = None
            for p in range(len(prev_ids)):
                if len(parent_children.get(p, [])) != 1:
                    continue
                if D[p, c] > DIV_PARENT_DIST_UM:
                    continue
                sister = parent_children[p][0]
                sister_dist = float(np.linalg.norm(C[c] - C[sister]))
                if sister_dist <= DIV_SISTER_DIST_UM:
                    # Score: parent proximity dominates; sister proximity guards against random nearby cells.
                    score = float(D[p, c] + 0.25 * sister_dist)
                    if best is None or score < best[0]:
                        best = (score, p)
            if best is not None:
                _, p = best
                edges.append((int(prev_ids[p]), int(curr_ids[c])))
                parent_children[p].append(int(c))
                matched_curr.add(int(c))

    return edges


def connected_components_from_edges(node_ids: Iterable[int], edges: List[Tuple[int, int]]) -> Dict[int, int]:
    """Return node_id -> component_id for undirected components of the predicted graph."""
    node_ids = list(node_ids)
    adj = {int(n): [] for n in node_ids}
    for s, t in edges:
        if s in adj and t in adj:
            adj[s].append(t)
            adj[t].append(s)

    comp = {}
    cid = 0
    for n in node_ids:
        if n in comp:
            continue
        cid += 1
        q = deque([n])
        comp[n] = cid
        while q:
            u = q.popleft()
            for v in adj[u]:
                if v not in comp:
                    comp[v] = cid
                    q.append(v)
    return comp


def prune_dataset_nodes(
    node_rows: List[dict],
    edge_rows: List[dict],
    node_scores: Dict[int, float],
) -> Tuple[List[dict], List[dict], Dict[str, int]]:
    """Remove isolated detections that contribute node penalty but no edge evidence."""
    if not PRUNE_ISOLATED_NODES or not node_rows:
        return node_rows, edge_rows, {'removed_isolated': 0, 'kept_nodes': len(node_rows), 'kept_edges': len(edge_rows)}

    all_ids = [int(r['node_id']) for r in node_rows]
    incident = set()
    edges = [(int(e['source_id']), int(e['target_id'])) for e in edge_rows]
    for s, t in edges:
        incident.add(s)
        incident.add(t)

    keep = set(incident)

    if KEEP_STRONG_ISOLATED:
        scores = np.array([node_scores.get(n, 0.0) for n in all_ids], dtype=np.float32)
        if len(scores):
            floor = float(np.quantile(scores, STRONG_ISOLATED_QUANTILE))
            for n in all_ids:
                if node_scores.get(n, 0.0) >= floor:
                    keep.add(n)

    kept_nodes = [r for r in node_rows if int(r['node_id']) in keep]
    kept_ids = {int(r['node_id']) for r in kept_nodes}
    kept_edges = [e for e in edge_rows if int(e['source_id']) in kept_ids and int(e['target_id']) in kept_ids]

    return kept_nodes, kept_edges, {
        'removed_isolated': len(node_rows) - len(kept_nodes),
        'kept_nodes': len(kept_nodes),
        'kept_edges': len(kept_edges),
    }


## 4 · Build `submission.csv`

The notebook emits both required row types: `node` rows with integer centroids and `edge` rows linking `source_id → target_id`. Node IDs are unique inside each dataset, which is sufficient for the competition metric.

In [5]:

def node_row(dataset: str, node_id: int, t: int, zyx: Sequence[int]) -> dict:
    z, y, x = [int(v) for v in zyx]
    return {
        'dataset': dataset,
        'row_type': 'node',
        'node_id': int(node_id),
        't': int(t),
        'z': z,
        'y': y,
        'x': x,
        'source_id': -1,
        'target_id': -1,
    }


def edge_row(dataset: str, source_id: int, target_id: int) -> dict:
    return {
        'dataset': dataset,
        'row_type': 'edge',
        'node_id': -1,
        't': -1,
        'z': -1,
        'y': -1,
        'x': -1,
        'source_id': int(source_id),
        'target_id': int(target_id),
    }


def process_dataset(dataset: str) -> Tuple[List[dict], Dict[str, object]]:
    zarr_path = TEST_DIR / f'{dataset}.zarr'
    array_dir, shape, dtype, chunk_shape = read_array_meta(zarr_path)
    T, Z, Y, X = shape

    print(f'\n[{dataset}] shape={shape}, dtype={dtype}, chunk={chunk_shape}')
    t0 = time.time()

    node_rows: List[dict] = []
    edge_rows: List[dict] = []
    node_scores: Dict[int, float] = {}

    prev_ids: List[int] = []
    prev_xyz = np.empty((0, 3), dtype=np.int32)
    prev_count: Optional[int] = None
    next_node_id = 1

    frame_counts = []
    division_count = 0

    for t in range(T):
        vol = load_volume(array_dir, shape, dtype, t)
        coords, scores = detect_cells(vol, prev_count=prev_count)
        del vol
        gc.collect()

        # Stable ordering improves reproducibility and notebook diffs.
        if len(coords):
            order = np.lexsort((coords[:, 2], coords[:, 1], coords[:, 0]))
            coords = coords[order]
            scores = scores[order]

        curr_ids = list(range(next_node_id, next_node_id + len(coords)))
        next_node_id += len(coords)

        for nid, zyx, score in zip(curr_ids, coords, scores):
            node_rows.append(node_row(dataset, nid, int(t), zyx))
            node_scores[int(nid)] = float(score)

        if t > 0:
            links = link_frames(prev_ids, prev_xyz, curr_ids, coords)
            for s, u in links:
                edge_rows.append(edge_row(dataset, s, u))
            src_counts = Counter(s for s, _ in links)
            division_count += sum(1 for c in src_counts.values() if c >= 2)

        prev_ids = curr_ids
        prev_xyz = coords
        prev_count = len(coords)
        frame_counts.append(len(coords))

        if (t + 1) % 10 == 0 or t == T - 1:
            print(f'  frame {t+1:>3}/{T}: nodes={len(coords):>4}, mean10={np.mean(frame_counts[-10:]):.1f}, edges={len(edge_rows)}')

    before_nodes, before_edges = len(node_rows), len(edge_rows)
    node_rows, edge_rows, prune_stats = prune_dataset_nodes(node_rows, edge_rows, node_scores)

    stats = {
        'dataset': dataset,
        'shape': shape,
        'nodes_before_prune': before_nodes,
        'edges_before_prune': before_edges,
        'nodes_after_prune': len(node_rows),
        'edges_after_prune': len(edge_rows),
        'removed_isolated': prune_stats['removed_isolated'],
        'division_edges_est': division_count,
        'count_min': int(min(frame_counts)) if frame_counts else 0,
        'count_max': int(max(frame_counts)) if frame_counts else 0,
        'count_mean': float(np.mean(frame_counts)) if frame_counts else 0.0,
        'seconds': time.time() - t0,
    }

    print(
        f'[{dataset}] done in {stats["seconds"]/60:.1f} min | '
        f'nodes {before_nodes}->{len(node_rows)}, edges {before_edges}->{len(edge_rows)}, '
        f'isolated_removed={stats["removed_isolated"]}, count_range=({stats["count_min"]}, {stats["count_max"]})'
    )
    return node_rows + edge_rows, stats


def build_submission() -> pd.DataFrame:
    datasets = list_datasets()
    all_rows: List[dict] = []
    all_stats = []
    t0 = time.time()

    for i, ds in enumerate(datasets, 1):
        rows, stats = process_dataset(ds)
        all_rows.extend(rows)
        all_stats.append(stats)
        print(f'Progress {i}/{len(datasets)} | cumulative rows={len(all_rows):,} | elapsed={(time.time()-t0)/60:.1f} min')

    sub = pd.DataFrame(all_rows)
    expected_cols = ['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
    sub = sub[expected_cols]
    sub.index.name = 'id'
    sub.to_csv(OUTPUT_PATH)

    print(f'\nWrote {OUTPUT_PATH} with {len(sub):,} rows in {(time.time()-t0)/60:.1f} min')
    print(sub['row_type'].value_counts())
    display(pd.DataFrame(all_stats))
    display(sub.head())
    return sub


## 5 · Validation

These assertions catch the common Kaggle submission failures before committing: missing datasets, wrong columns, negative node coordinates, and dangling edge references.

In [6]:

def validate_submission(sub: pd.DataFrame) -> None:
    expected_cols = ['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
    assert list(sub.columns) == expected_cols, f'Wrong columns: {list(sub.columns)}'
    assert len(sub) > 0, 'Empty submission'
    assert set(sub['row_type'].unique()).issubset({'node', 'edge'}), 'Invalid row_type values'

    datasets = set(dataset_names_from_test_dir(TEST_DIR))
    assert datasets.issubset(set(sub['dataset'].unique())), 'Every test dataset must appear in submission'

    nodes = sub[sub.row_type == 'node']
    edges = sub[sub.row_type == 'edge']

    assert (nodes[['node_id', 't', 'z', 'y', 'x']] >= 0).all().all(), 'Node ids/time/coords must be non-negative'
    assert (nodes[['source_id', 'target_id']] == -1).all().all(), 'Node source/target must be -1'
    assert (edges[['node_id', 't', 'z', 'y', 'x']] == -1).all().all(), 'Edge node/time/coords must be -1'
    assert (edges[['source_id', 'target_id']] >= 0).all().all(), 'Edge references must be non-negative'

    for ds, grp in sub.groupby('dataset'):
        node_ids = set(grp.loc[grp.row_type == 'node', 'node_id'].astype(int))
        e = grp[grp.row_type == 'edge']
        assert len(node_ids) > 0, f'{ds}: no node rows'
        assert e['source_id'].astype(int).isin(node_ids).all(), f'{ds}: dangling source_id'
        assert e['target_id'].astype(int).isin(node_ids).all(), f'{ds}: dangling target_id'

    print('All validation checks passed ✅')


submission = build_submission()
validate_submission(submission)


Found 4 test datasets: ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']

[44b6_0113de3b] shape=(100, 64, 256, 256), dtype=uint16, chunk=(1, 64, 256, 256)
  frame  10/100: nodes= 143, mean10=140.0, edges=1147
  frame  20/100: nodes= 143, mean10=137.6, edges=2406
  frame  30/100: nodes= 150, mean10=140.8, edges=3684
  frame  40/100: nodes= 155, mean10=150.7, edges=5057
  frame  50/100: nodes= 149, mean10=144.1, edges=6359
  frame  60/100: nodes= 152, mean10=151.1, edges=7723
  frame  70/100: nodes= 148, mean10=154.3, edges=9135
  frame  80/100: nodes= 138, mean10=147.5, edges=10478
  frame  90/100: nodes= 144, mean10=149.1, edges=11835
  frame 100/100: nodes= 140, mean10=144.9, edges=13170
[44b6_0113de3b] done in 0.4 min | nodes 14601->14268, edges 13170->13170, isolated_removed=333, count_range=(130, 164)
Progress 1/4 | cumulative rows=27,438 | elapsed=0.4 min

[44b6_0b24845f] shape=(100, 64, 256, 256), dtype=uint16, chunk=(1, 64, 256, 256)
  frame  10/100: nodes=  6

,dataset,shape,nodes_before_prune,edges_before_prune,nodes_after_prune,edges_after_prune,removed_isolated,division_edges_est,count_min,count_max,count_mean,seconds
0,44b6_0113de3b,"(100, 64, 256, 256)",14601,13170,14268,13170,333,2,130,164,146.01,21.776580
1,44b6_0b24845f,"(100, 64, 256, 256)",7955,6025,7266,6025,689,0,60,101,79.55,20.052620
2,6bba_05b6850b,"(100, 64, 256, 256)",3664,3402,3598,3402,66,0,29,45,36.64,17.332459
3,6bba_05db0fb1,"(100, 64, 256, 256)",16498,14350,16008,14350,490,1,118,221,164.98,22.001842


,dataset,row_type,node_id,t,z,y,x,source_id,target_id
id,,,,,,,,,
0,44b6_0113de3b,node,1,0,0,6,53,-1,-1
1,44b6_0113de3b,node,2,0,1,63,58,-1,-1
2,44b6_0113de3b,node,3,0,1,89,90,-1,-1
3,44b6_0113de3b,node,4,0,2,94,59,-1,-1
4,44b6_0113de3b,node,5,0,2,146,47,-1,-1


All validation checks passed ✅


## Practical tuning notes

This V2 default is designed as the next safe submission after a 0.581 public score. The next leaderboard sweeps should be surgical:

- If the score drops: set `BIOHUB_PRUNE_ISOLATED_NODES=0` or `BIOHUB_THRESH_REL=0.30`.
- If node count still looks too high: try `BIOHUB_THRESH_REL=0.38` and `BIOHUB_BORDER_KEEP_QUANTILE=0.80`.
- If division false positives appear costly: set `BIOHUB_DETECT_DIVISIONS=0` for an A/B test.
- If edge recall is weak: try `BIOHUB_MAX_LINK_DIST_UM=12.0`.

For a top-tier solution, the next real jump is train-label calibration or a pretrained 3D cell detector attached as a Kaggle Dataset. This notebook stays classical, deterministic, no-internet, and fast enough for repeated leaderboard iteration.